# System Q&A oparty o RAG (Retrieval-Augmented Generation) z użyciem HuggingFace Embeddings

W tym notatniku krok po kroku zbudujemy system odpowiadający na pytania na podstawie dokumentów (np. raporty, earnings calls itp.), wykorzystując podejście RAG.

---

## 1. Wybór modelu AI

### Dlaczego HuggingFace Embeddings + RAG?

- **Embeddings** (osadzenia) to wektorowa reprezentacja tekstu, która pozwala na porównywanie podobieństwa semantycznego między fragmentami tekstu.
- **HuggingFace Embeddings** są darmowe, open-source i dobrze sprawdzają się w zadaniach wyszukiwania informacji (retrieval). Dobrą alternatywą będzie OpenAI, ale wymaga on płatnego planu.
- **RAG** (Retrieval-Augmented Generation) to architektura, w której najpierw wyszukujemy najbardziej pasujące fragmenty dokumentów (retrieval), a następnie generujemy odpowiedź na pytanie użytkownika z użyciem dużego modelu językowego (LLM), bazując na tych fragmentach.

### Schemat działania systemu:

1. **Indeksowanie dokumentów** – dzielimy dokumenty na fragmenty i zamieniamy je na wektory (embeddings).
2. **Wyszukiwanie** – dla zadanego pytania generujemy embedding i szukamy najbardziej podobnych fragmentów w bazie wektorowej.
3. **Generowanie odpowiedzi** – przekazujemy znalezione fragmenty oraz pytanie do modelu językowego, który generuje odpowiedź.

---

## 2. Wymagane pakiety

Aby zrealizować ten projekt, będziemy potrzebować m.in.:
- `sentence-transformers` – darmowe modele embeddingów
- `langchain` – framework do budowy systemów RAG
- `chromadb` – baza wektorowa (do przechowywania embeddings). Alternatywa dla chromadb to Pinecone, Azure AI Search itp.
- `PyPDFLoader` – ładowanie i dzielenie dokumentów PDF
- `gradio` – prosty interfejs użytkownika (opcjonalnie)

Instalacja (w terminalu):
```bash
pip install sentence-transformers langchain chromadb pypdf gradio
```

---

## 3. Kolejne kroki

W kolejnych sekcjach:
- Przygotujemy dokumenty i wygenerujemy embeddings.
- Zbudujemy bazę wektorową.
- Zaimplementujemy wyszukiwanie i generowanie odpowiedzi.
- Stworzymy prosty interfejs użytkownika.

---

Następnym etapem jest przygotowanie i wczytanie dokumentów, które będą źródłem wiedzy dla systemu Q&A.  
Aby rozpocząć, wrzuć przykładowe pliki PDF (np. raporty, earnings calls) do folderu `../data/`.  
System automatycznie wczyta wszystkie pliki PDF znajdujące się w tym katalogu.

Poniżej jest fragment przykładowej procedury, która dla sprawdzenia poprawności działania wczytuje tylko jeden plik PDF:


In [ ]:
# Wczytywanie i dzielenie dokumentów PDF na fragmenty

from langchain.document_loaders import PyPDFLoader

# Ścieżka do jednego pliku PDF
pdf_path = "../data/raport.pdf"  # zmień na swój plik PDF

# Wczytaj dokument
loader = PyPDFLoader(pdf_path)
documents = loader.load_and_split()

print(f"Liczba fragmentów: {len(documents)}")
print("Przykładowy fragment:", documents[0].page_content[:500])

### 4. Wczytywanie wszystkich dokumentów PDF z katalogu

Teraz zmodyfikujemy kod, aby wczytać wszystkie pliki PDF znajdujące się w katalogu `../data/`. Użyjemy `glob` do znalezienia plików i przetworzymy je w pętli.

In [ ]:
# Wczytywanie wszystkich dokumentów PDF z katalogu

from langchain.document_loaders import PyPDFLoader
import glob
import os

# Ścieżka do katalogu z plikami PDF
data_dir = "../data/"
all_documents = []

# Znajdź wszystkie pliki PDF w katalogu
pdf_files = glob.glob(os.path.join(data_dir, "*.pdf"))

print(f"Znaleziono {len(pdf_files)} plików PDF do przetworzenia.")

# Wczytaj i podziel każdy dokument
for pdf_path in pdf_files:
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load_and_split()
        # Dodaj informację o źródle do metadanych
        for doc in documents:
            doc.metadata["source_file"] = os.path.basename(pdf_path)
        all_documents.extend(documents)
        print(f"Przetworzono: {os.path.basename(pdf_path)}, liczba fragmentów: {len(documents)}")
    except Exception as e:
        print(f"Błąd podczas przetwarzania pliku {pdf_path}: {e}")

print(f"\nŁączna liczba fragmentów ze wszystkich dokumentów: {len(all_documents)}")
if all_documents:
    print("Przykładowy fragment (pierwszy):", all_documents[0].page_content[:200])
    print("Źródło pierwszego fragmentu:", all_documents[0].metadata.get('source_file', 'Brak informacji o źródle'))

### 5. Generowanie embeddings i budowa bazy wektorowej

W tej sekcji wygenerujemy embeddings (wektorowe reprezentacje tekstu) dla wszystkich fragmentów dokumentów za pomocą Sentence Transformers (HuggingFace Embeddings). Następnie zapiszemy te wektory w lokalnej bazie wektorowej ChromaDB, co pozwoli nam na efektywne wyszukiwanie podobnych fragmentów podczas odpowiadania na pytania.

ChromaDB jest lekką bazą wektorową, która przechowuje wektory lokalnie na dysku, co umożliwia szybkie wyszukiwanie semantyczne bez konieczności korzystania z zewnętrznych usług.

In [ ]:
# Generowanie embeddings i budowa bazy wektorowej

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Sprawdzenie, czy mamy jakiekolwiek dokumenty do przetworzenia
if len(all_documents) > 0:
    
    # Inicjalizacja modelu embeddings HuggingFace (np. all-MiniLM-L6-v2)
    embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    # Tworzenie bazy wektorowej ChromaDB z dokumentów
    persist_directory = '../db_chroma'
    
    print("Generowanie embeddings i budowanie bazy wektorowej. To może potrwać kilka minut...")
    
    vectorstore = Chroma.from_documents(
        documents=all_documents,
        embedding=embeddings_model,
        persist_directory=persist_directory
    )
    
    # Zapisanie bazy na dysku
    vectorstore.persist()
    
    print(f"\n✅ Baza wektorowa została utworzona i zapisana w katalogu: {persist_directory}")
    print(f"Liczba zaindeksowanych fragmentów: {len(all_documents)}")
else:
    print("\n❌ Brak dokumentów do przetworzenia. Nie można utworzyć bazy wektorowej.")

### 6. Wyszukiwanie podobnych fragmentów na podstawie pytania użytkownika

W tej sekcji załadujemy bazę wektorową ChromaDB i zaimplementujemy funkcję, która dla zadanego pytania znajdzie najbardziej podobne fragmenty dokumentów. Dzięki temu będziemy mogli przekazać do modelu językowego tylko te fragmenty, które są najbardziej istotne dla odpowiedzi na pytanie użytkownika.

In [ ]:
# Wyszukiwanie podobnych fragmentów na podstawie pytania użytkownika

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Ścieżka do bazy wektorowej
persist_directory = '../db_chroma'

# Inicjalizacja modelu embeddings (taki sam jak przy budowie bazy)
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Załaduj bazę wektorową
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings_model
)

def search_similar_documents(query, k=5):
    """Zwraca k najbardziej podobnych fragmentów do zapytania użytkownika."""
    results = vectorstore.similarity_search(query, k=k)
    return results

# Przykład użycia:
user_question = "Jakie modele telefonów komórkowych testowano w raporcie z dnia 30.11.2023?"

top_docs = search_similar_documents(user_question, k=3)
for i, doc in enumerate(top_docs, 1):
    print(f"\nFragment {i}:\n{doc.page_content[:500]}")

### 7. Generowanie odpowiedzi na pytania użytkownika z użyciem modelu językowego HuggingFace

W tej sekcji wykorzystujemy model językowy z Hugging Face Hub do generowania odpowiedzi na pytania użytkownika na podstawie znalezionych fragmentów dokumentów.

- Najpierw ustawiamy token API do Hugging Face (`HUGGINGFACEHUB_API_TOKEN`), który jest wymagany do korzystania z modeli hostowanych w chmurze.
- Inicjalizujemy model językowy (`HuggingFaceHub`) – w przykładzie używamy modelu `zephyr-7b-beta`, który jest lekki (poniżej 10GB) i dobrze sprawdza się w zadaniach generowania tekstu.
- Tworzymy łańcuch RAG (`RetrievalQA`), który automatycznie pobiera najbardziej pasujące fragmenty z bazy wektorowej i przekazuje je do modelu językowego.
- Przykład użycia: zadanie pytania do systemu, wyszukiwanie podobnych fragmentów oraz generowanie odpowiedzi przez model.

Dzięki temu system automatycznie łączy wyszukiwanie wiedzy w dokumentach z generowaniem odpowiedzi przez LLM.

**Uwaga:** Przed uruchomieniem tego fragmentu kodu należy podać własny token API Hugging Face w miejscu `"hf_TWOJ_TOKEN"`.
Hugging Face posiada limity darmowych zapytań miesięcznie, po przekroczeniu limitu wymagany jest plan PRO lub pobranie modelu i generowanie odpowiedzi lokalnie.

In [ ]:
from langchain.llms import HuggingFaceHub
from langchain.chains import RetrievalQA

import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_TWOJ_TOKEN"  # <-- Wprowadź swój token API Hugging Face

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",  # model <10GB (ze względu na limit HF model musi być mniejszy niż 10GB)
    model_kwargs={"temperature": 0.2, "max_new_tokens": 512}
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

# Przykład użycia: zadanie pytania do modelu na podstawie dokumentów z bazy wektorowej
user_question = "Zrób streszczenie raportu z dnia 30.11.2023."

# Wyszukiwanie podobnych dokumentów
top_docs = search_similar_documents(user_question, k=3)

answer = qa_chain.run(user_question)
print("Odpowiedź modelu:\n", answer)

### 8. Interfejs użytkownika z Gradio

W tej sekcji tworzymy prosty interfejs webowy za pomocą biblioteki Gradio, który umożliwia zadawanie pytań do systemu Q&A bezpośrednio z poziomu przeglądarki.

- Funkcja `rag_qa_interface` przyjmuje pytanie użytkownika, przekazuje je do łańcucha RAG i zwraca wygenerowaną odpowiedź.
- Interfejs Gradio udostępnia pole tekstowe do wpisania pytania oraz pole z odpowiedzią.
- Po uruchomieniu kodu pojawi się lokalny adres URL, pod którym można korzystać z systemu Q&A w przeglądarce.

Dzięki temu nawet osoby nietechniczne mogą łatwo korzystać z systemu, zadając pytania dotyczące załadowanych dokumentów PDF.

In [ ]:
import gradio as gr

def rag_qa_interface(question):
    answer = qa_chain.run(question)
    return answer

gr.Interface(
    fn=rag_qa_interface,
    inputs=gr.Textbox(lines=2, label="Twoje pytanie"),
    outputs=gr.Textbox(label="Odpowiedź"),
    title="System Q&A oparty o RAG (HuggingFace)",
    description="Zadaj pytanie dotyczące załadowanych dokumentów PDF. Odpowiedź zostanie wygenerowana na podstawie znalezionych fragmentów."
).launch()